# Gene enrichment, split by immediate vs. previous response gene sets
I've done enrichment analyses for some of the phase 2 comparisons (see [gsea](https://github.com/jgmcdonough/CE24_RNA-seq/blob/main/analysis/diff_expression/phase2_v_phase2/gsea_p2.v.p2.ipynb) and [ora](https://github.com/jgmcdonough/CE24_RNA-seq/blob/main/analysis/diff_expression/phase2_v_phase2/ora_p2.v.p2.ipynb)), but I have over 50 pairwise comparisons and it's hard to go through results by looking at everything at once. 

In a [previous notebook](https://github.com/jgmcdonough/CE24_RNA-seq/blob/main/analysis/diff_expression/phase2_v_phase2/upset_p2.v.p2.ipynb) I compared DEGs between pairwise comparisons that differed by their most recent exposure (same phase 1, different phase 2) or by their previous exposure (different phase 2, same phase 1). The list of comparisons can be found [here](https://github.com/jgmcdonough/CE24_RNA-seq/blob/main/analysis/diff_expression/phase2_v_phase2/imm_prev_datasets.csv).

In this notebook, I'm going to dig into the gene annotation and enrichment for these datasets to think more broadly about what processes are enriched due to the previous vs. immediate response to enviornmental stress.

## 0. load libraries

In [25]:
library(tidyverse)

# enrichment analysis 
library(clusterProfiler) # for GSEA()
library(enrichplot) # for enrichment visuals
library(GO.db) # for gene ontology database
library(fgsea) # fast pre-ranked GSEA

# visualizing
library(patchwork) # add multiple plots together
library(RColorBrewer) 

## 1. read.csvs

### comparisons of interest

In [8]:
im.prev <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/imm_prev_datasets.csv')

dim(im.prev)
head(im.prev)

[1] 96 16

,dataset,total_genes,deg_numb,phase1,phase2,expr_high,comparison,DEG_count,results,stress_set,is_outlier,xaxis,comp1,comp2,lower,diff_treatment
,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>
1,bc_bb,204,31,B,C,BC,BC vs. BB,204,Immediate Exposure,Both,FALSE,"Same phase 1, different phase 2",bc,bb,bc vs. bb,Control
2,bc_bb,204,173,B,B,BB,BC vs. BB,204,Immediate Exposure,Both,FALSE,"Same phase 1, different phase 2",bc,bb,bc vs. bb,Both
3,bc_cc,281,187,B,C,BC,BC vs. CC,281,Previous Exposure,Control,TRUE,"Different phase 1, same phase 2",bc,cc,bc vs. cc,Both
4,bc_cc,281,94,C,C,CC,BC vs. CC,281,Previous Exposure,Control,TRUE,"Different phase 1, same phase 2",bc,cc,bc vs. cc,Control
5,bc_hc,134,35,B,C,BC,BC vs. HC,134,Previous Exposure,Control,FALSE,"Different phase 1, same phase 2",bc,hc,bc vs. hc,Both
6,bc_hc,134,99,H,C,HC,BC vs. HC,134,Previous Exposure,Control,FALSE,"Different phase 1, same phase 2",bc,hc,bc vs. hc,Hypoxic


### deseq results

In [9]:
# get list of files
files <- list.files(
    path = '/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq/',
    pattern = '\\.csv$',
    full.names = TRUE
    )

head(files)

[1] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bb_cc.csv"
[2] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bc_bb.csv"
[3] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bc_cc.csv"
[4] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bc_hc.csv"
[5] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bc_wc.csv"
[6] "/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bh_bb.csv"

In [10]:
names(files) <- tools::file_path_sans_ext(basename(files))
df_list <- lapply(files, read.csv)
names(df_list)
length(df_list)

[1] "bb_cc"        "bc_bb"        "bc_cc"        "bc_hc"        "bc_wc"       
 [6] "bh_bb"        "bh_bc"        "bh_bw"        "bh_ch"        "bh_hh"       
[11] "bw_bb"        "bw_bc"        "bw_cw"        "bw_ww"        "cb_bb"       
[16] "cb_bc"        "cb_cc"        "cb_ch"        "cb_cw"        "ch_cc"       
[21] "ch_hc"        "cw_cc"        "cw_ch"        "cw_wc"        "hb_bb"       
[26] "hb_bh"        "hb_cb"        "hb_hc"        "hb_hh"        "hb_hw"       
[31] "hc_cc"        "hc_hh"        "hh_cc"        "hh_ch"        "hw_bw"       
[36] "hw_cw"        "hw_hc"        "hw_hh"        "hw_ww"        "vst_filtered"
[41] "wb_bb"        "wb_bw"        "wb_cb"        "wb_hb"        "wb_wc"       
[46] "wb_wh"        "wb_ww"        "wc_cc"        "wc_hc"        "wc_ww"       
[51] "wh_bh"        "wh_ch"        "wh_hh"        "wh_hw"        "wh_wc"       
[56] "wh_ww"        "ww_cc"        "ww_cw"

[1] 58

keep only the comparisons that match the dataset of interest

In [12]:
head(basename(files))
head(im.prev$dataset)

[1] "bb_cc.csv" "bc_bb.csv" "bc_cc.csv" "bc_hc.csv" "bc_wc.csv" "bh_bb.csv"

[1] "bc_bb" "bc_bb" "bc_cc" "bc_cc" "bc_hc" "bc_hc"

In [15]:
wanted_files <- files[tools::file_path_sans_ext(basename(files)) %in% im.prev$dataset]
head(wanted_files)

bc_bb 
"/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bc_bb.csv" 
                                                                                                                                               bc_cc 
"/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bc_cc.csv" 
                                                                                                                                               bc_hc 
"/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bc_hc.csv" 
                                                                                                                                               bc_wc 
"/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bc_wc.csv" 
                                                                                                                                               bh_bb 
"/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bh_bb.csv" 
                                                                                                                                               bh_bc 
"/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase2_v_phase2/deseq_res/prefilter_deseq//bh_bc.csv"

In [22]:
res_list <- lapply(wanted_files, data.table::fread)
names(res_list)
length(names(res_list))

[1] "bc_bb" "bc_cc" "bc_hc" "bc_wc" "bh_bb" "bh_bc" "bh_bw" "bh_ch" "bh_hh"
[10] "bw_bb" "bw_bc" "bw_cw" "bw_ww" "cb_bb" "cb_cc" "cb_ch" "cb_cw" "ch_cc"
[19] "cw_cc" "cw_ch" "hb_bb" "hb_cb" "hb_hc" "hb_hh" "hb_hw" "hc_cc" "hc_hh"
[28] "hh_ch" "hw_bw" "hw_cw" "hw_hc" "hw_hh" "hw_ww" "wb_bb" "wb_cb" "wb_hb"
[37] "wb_wc" "wb_wh" "wb_ww" "wc_cc" "wc_hc" "wc_ww" "wh_bh" "wh_ch" "wh_hh"
[46] "wh_wc" "wh_ww" "ww_cw"

[1] 48

split dfs by results (immediate vs. previous exposure)

In [24]:
groups <- split(im.prev$dataset, im.prev$results)          
res_by_group <- lapply(groups, function(ds) res_list[ds])

names(res_by_group) # this is a list of lists

[1] "Immediate Exposure" "Previous Exposure"

### gene annotations and ontology

In [5]:
# col1 = gene ID
# col2 = GO ID 
gene2go <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cvirginica_genome/annotations/newRef_geneGO.csv')
head(gene2go)

,gene,Gene.Ontology.IDs
,<chr>,<chr>
1,LOC111099029,GO:0005261;GO:0005886;GO:0030001;GO:0098655
2,LOC111099032,GO:0004930;GO:0005886;GO:0007186
3,LOC111099037,GO:0000978;GO:0000981;GO:0006355
4,LOC111099039,GO:0004930;GO:0005886;GO:0007189
5,LOC111099040,GO:0005515;GO:0007169;GO:0035556
6,LOC111099041,GO:0016020;GO:0022857;GO:0055085


re-format for correct input for `GSEA()` - two columns, one for GO term and one for gene ID

In [6]:
term2gene <- gene2go %>%
  mutate(GO_terms = strsplit(Gene.Ontology.IDs, ",\\s*|;\\s*|`")) %>%  # Split by comma, semicolon, or backtick
  unnest(GO_terms) %>%
  filter(grepl("^GO:", GO_terms)) %>%  # Keep only valid GO terms
  dplyr::select(term = GO_terms, gene = gene)

class(term2gene)
str(term2gene)
head(term2gene)

[1] "tbl_df"     "tbl"        "data.frame"

tibble [39,588 × 2] (S3: tbl_df/tbl/data.frame)
 $ term: chr [1:39588] "GO:0005261" "GO:0005886" "GO:0030001" "GO:0098655" ...
 $ gene: chr [1:39588] "LOC111099029" "LOC111099029" "LOC111099029" "LOC111099029" ...


term,gene
<chr>,<chr>
GO:0005261,LOC111099029
GO:0005886,LOC111099029
GO:0030001,LOC111099029
GO:0098655,LOC111099029
GO:0004930,LOC111099032
GO:0005886,LOC111099032


get term names for GO IDs

In [7]:
# Extract GO term descriptions
go_terms <- unique(term2gene$term)

# Get descriptions from GO.db
term2name <- data.frame(
  term = go_terms,
  name = sapply(go_terms, function(x) {
    tryCatch({
      Term(GOTERM[[x]])
    }, error = function(e) {
      NA_character_
    })
  })
)

# Remove NAs
term2name <- term2name[!is.na(term2name$name), ]

# View
head(term2name)    

,term,name
,<chr>,<chr>
GO:0005261,GO:0005261,monoatomic cation channel activity
GO:0005886,GO:0005886,plasma membrane
GO:0030001,GO:0030001,metal ion transport
GO:0098655,GO:0098655,monoatomic cation transmembrane transport
GO:0004930,GO:0004930,G protein-coupled receptor activity
GO:0007186,GO:0007186,G protein-coupled receptor signaling pathway


## 2. GSEA
I want to compile enrichment results for response due to immediate treatment vs previous treatment. 

below is a code from Claude that I haven't tested yet

In [ ]:
# 3. Build a ranked vector from one DESeq2 result
make_ranks <- function(df) {
  df <- df[!is.na(df$stat) & !is.na(df$gene), ]   # 'gene' = your gene ID column
  df <- df[!duplicated(df$gene), ]
  sort(setNames(df$stat, df$gene), decreasing = TRUE)
}

# 4. Run GSEA on every dataset in every group
run_gsea <- function(df) {
  fgseaMultilevel(pathways, make_ranks(df), minSize = 15, maxSize = 500)
}

gsea_res <- lapply(res_by_group, function(g) lapply(g, run_gsea))

# 5. Combine into one table with dataset and group labels
gsea_df <- bind_rows(lapply(names(gsea_res), function(g) {
  bind_rows(gsea_res[[g]], .id = "dataset") |> mutate(group = g)
}))